# Data Explorer
Inspect the downloaded price data as an interactive table.
Use the **Open in Data Wrangler** button (appears above any DataFrame output) for a full spreadsheet view.

In [ ]:
import pandas as pd
from pathlib import Path
from IPython.display import display

pd.set_option("display.max_columns", None)   # show all columns
pd.set_option("display.float_format", "{:.2f}".format)

DATA_DIR = Path("../data/raw")

## Load Data

In [ ]:
# List available files
available = list(DATA_DIR.glob("*.parquet")) + list(DATA_DIR.glob("*.csv"))
print("Files in data/raw:")
for f in available:
    print(f"  {f.name}  ({f.stat().st_size / 1e6:.1f} MB)")

In [ ]:
# Load the universe file (long format: one row per stock × day)
path = DATA_DIR / "universe.parquet"
df = pd.read_parquet(path)

print(f"Shape      : {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Date range : {df['date'].min().date()} → {df['date'].max().date()}")
print(f"Tickers    : {df['ticker'].nunique():,}")
print(f"Sources    : {df.groupby('source')['ticker'].nunique().to_dict()}")
print()
display(df.head(10))

In [ ]:
# Column types and basic stats
print("Schema:")
print(df.dtypes)
print()
display(df.describe())

## Filter and Sort

In [ ]:
# Per-ticker summary: trading days, price range, avg close
summary = (
    df.groupby(["ticker", "source"])["close"]
    .agg(trading_days="count", min_price="min", max_price="max", mean_close="mean")
    .sort_values("mean_close", ascending=False)
    .reset_index()
)
print(f"Tickers with data: {len(summary)}")
display(summary)

In [ ]:
# Filter: specific tickers and/or date range
TICKERS = ["AAPL", "MSFT", "NVDA", "TSM", "NVO"]  # change as needed
START   = "2025-01-01"
END     = "2025-12-31"

subset = df[
    df["ticker"].isin(TICKERS) &
    df["date"].between(START, END)
].sort_values(["ticker", "date"])

print(f"Filtered: {len(subset):,} rows | {subset['ticker'].nunique()} tickers")
display(subset)

## Styled Table
Color gradient highlights high (green) and low (red) closing prices per column.

In [ ]:
# Pivot to wide for styled view: rows = date, cols = tickers
pivot = (
    subset.pivot(index="date", columns="ticker", values="close")
    .tail(20)
)

styled = (
    pivot.style
    .background_gradient(cmap="RdYlGn", axis=0)
    .format("{:.2f}")
    .set_caption("Adjusted Close Prices (red = low, green = high per column)")
)
display(styled)

## Open in Data Wrangler (full spreadsheet view)
After running any cell that outputs a DataFrame, a **"Open in Data Wrangler"** button appears in the output toolbar.  
Click it for a resizable, sortable, filterable spreadsheet — no extra code needed.

You can also right-click any `.parquet` file in the Explorer panel → **Open in Data Wrangler** to view it directly.